# CPSC 483 — Classwork: Linear Regression on Auto MPG
**Slide 22 | Dr. Panangadan | June 2, 2026**

Goal: model `mpg` (city-cycle fuel efficiency) given `displacement` (engine size in cubic inches).

## Setup — imports & load data

The dataset has **398 rows**. After dropping the 6 rows with missing `horsepower`, we work with **392 rows**.

> **Upload `autompg.csv` from Canvas before running this cell.**

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Load — upload autompg.csv from Canvas first
df = pd.read_csv('autompg.csv').dropna()
print('Shape:', df.shape)   # should be (392, 9)
df.head()

## Q1 — Identify the variables

- **Dependent variable (Y):** `mpg` — the value we are predicting (continuous → regression task)
- **Independent variable (X):** `displacement` — the single explanatory/input feature

In [ ]:
X = df[['displacement']].values   # shape (392, 1) — 2D required by sklearn
y = df['mpg'].values               # shape (392,)  — 1D target

print(f'X (displacement): {X.shape}  range {X.min():.0f} – {X.max():.0f} cu.in.')
print(f'y (mpg):          {y.shape}  range {y.min():.1f} – {y.max():.1f} mpg')

## Q2 — Plot mpg vs. displacement

**Before running:** Based on domain knowledge (bigger engine → more fuel), should the trend be positive or negative?

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.4, color='steelblue', edgecolors='none')
plt.xlabel('Engine Displacement (cu. in.)')
plt.ylabel('MPG')
plt.title('Auto MPG — MPG vs. Engine Displacement')
plt.grid(True)
plt.tight_layout()
plt.show()

## Q3 — Fit the best-fit linear regression model

`LinearRegression()` solves the **Normal Equation** internally (via SVD/pseudoinverse).

**Before running:** Since the trend is negative, predict the sign of `coef_` (θ₁).

In [ ]:
lin_reg = LinearRegression()
lin_reg.fit(X, y)

theta_0 = lin_reg.intercept_
theta_1 = lin_reg.coef_[0]

print(f'θ₀ (intercept):         {theta_0:.4f}')
print(f'θ₁ (displacement coef): {theta_1:.4f}')

## Q3b — R² (Coefficient of Determination)

R² = SSR / SST = proportion of variance in `mpg` explained by `displacement`.

- R² = 1.0 → perfect fit  
- R² = 0.0 → no better than predicting the mean

**Before running:** The scatter plot looked fairly noisy. Guess a ballpark R² value.

In [ ]:
r2 = lin_reg.score(X, y)
print(f'R² = {r2:.4f}')
print(f'→ displacement alone explains {r2*100:.1f}% of the variance in mpg')

## Q4 — Overlay best-fit line over the scatter plot

In [ ]:
X_line = np.linspace(X.min(), X.max(), 300).reshape(-1, 1)
y_line = lin_reg.predict(X_line)

plt.figure(figsize=(8, 5))
plt.scatter(X, y, alpha=0.4, color='steelblue', edgecolors='none', label='Data')
plt.plot(X_line, y_line, 'r-', linewidth=2, label=f'Best fit  R²={r2:.3f}')
plt.xlabel('Engine Displacement (cu. in.)')
plt.ylabel('MPG')
plt.title('Auto MPG — Linear Regression Best Fit')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## Q5 — Prediction equation

Hint from the slide: assemble the equation from `intercept_` and `coef_`.

In [ ]:
print('Prediction equation:')
print(f'  mpg = {theta_0:.4f} + ({theta_1:.4f}) × displacement')
print(f'  mpg = {theta_0:.2f} − {abs(theta_1):.4f} × displacement')
print()
print(f'θ₁ = {theta_1:.4f}: each +1 cu.in. of displacement reduces mpg by {abs(theta_1):.4f}')

## Q6 — Predictions

**Before running — think it through:**
- Training data displacement range: **68 – 455 cu.in.**
- `250` is inside that range → **interpolation** → trustworthy
- `600` is outside that range → **extrapolation** → what happens when a line keeps going past the data?

In [ ]:
pred_250 = lin_reg.predict([[250]])[0]
pred_600 = lin_reg.predict([[600]])[0]

print(f'Predicted mpg @ displacement=250: {pred_250:.2f} mpg  ← interpolation (within training range)')
print(f'Predicted mpg @ displacement=600: {pred_600:.2f} mpg  ← extrapolation (max training = {X.max():.0f})')
print()
print('A negative mpg is physically impossible.')
print('This is the danger of extrapolation — the model has no data to anchor it.')

## Summary of Results

| Item | Value |
|---|---|
| Dependent variable | `mpg` |
| Independent variable | `displacement` |
| θ₀ (intercept) | 35.1206 |
| θ₁ (slope) | −0.0601 |
| R² | 0.6482 |
| Prediction @ displacement=250 | **20.11 mpg** ✅ |
| Prediction @ displacement=600 | **−0.91 mpg** ❌ (extrapolation) |    

**Key takeaway:** R² ≈ 0.65 means displacement explains ~65% of mpg variation — decent for one feature, but clearly not the whole story.